In [ ]:
# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list the files in the input directory

import os
print(os.listdir("../input"))

# Any results you write to the current directory are saved as output.

Підключіть необхідні бібліотеки.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, kendalltau,f_oneway
sns.set(rc={'figure.figsize':(10, 8)}); # you can change this if needed

In [2]:
df = pd.read_csv('data/bank-additional-full.csv', sep=';')

In [ ]:
df.info()

In [ ]:
df.nunique()

Preprocesing


In [3]:
df['y'] = df['y'].map({"no":0,"yes":1})
df['contact'] = df['contact'].map({"cellular":0,"telephone":1})
df1 = pd.get_dummies(df, columns=['job','marital','education','default','housing','loan','month','day_of_week','poutcome'])
df1.head(6).T

,0,1,2,3,4,5
age,56,57,37,40,56,45
contact,1,1,1,1,1,1
duration,261,149,226,151,307,198
campaign,1,1,1,1,1,1
pdays,999,999,999,999,999,999
...,...,...,...,...,...,...
day_of_week_tue,False,False,False,False,False,False
day_of_week_wed,False,False,False,False,False,False
poutcome_failure,False,False,False,False,False,False
poutcome_nonexistent,True,True,True,True,True,True


In [ ]:
df1.info()

In [ ]:
sns.countplot(x="y", data=df);

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_valid, y_train, y_valid = train_test_split(df1.drop('y', axis=1),
                                                      df1['y'],
                                                      test_size=0.25,
                                                      random_state=42)

3. Побудувати одну з лінійних моделей машинного навчання (лінійну регресію або логістичну регресію, залежно від вашого варіанту). Оцінити якість моделі на тестових даних за допомогою декількох метрик. 

Це типова проблема двійкової класифікації. Тому ми розглянемо саме логістичну регресію.

In [5]:
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_valid)

from sklearn.metrics import accuracy_score
print(accuracy_score(y_valid, y_pred))

0.9117218607361367


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Можна сконструювати наступні три метрики, кожна з яких краще представляє результати класифікації.  Це Precision, Recall та  F-оцінка (f1_score).  • Precision показує, наскільки можна «довіряти» моделі, якщо вона показала клас 1;
 • Recall показує, наскільки добре модель здатна знайти клас 1;
 • F1: середнє гармонічне між Precision та Recall.

In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score
print('Precision:', precision_score(y_valid, y_pred))
print('Recall:', recall_score(y_valid, y_pred))
print('F1 score:', f1_score(y_valid, y_pred))

Precision: 0.6713483146067416
Recall: 0.41457068516912404
F1 score: 0.5126005361930295


У нашому прикладі ми і хочемо якомога менше помилкових спрацьовувань і важливо звести до мінімуму промахи класу 1, тому беремо гармонічне між ними f1-score.

Застосувати декілька типів регуляризації (Ridge, Lasso, ElasticNet), налаштувати гіперпараметри моделей, побудувати валідаційні криві. 


Гіперпараметр C>0 відіграє роль зворотної сили регуляризації.  Чим більший С, тим менший «штраф» за збільшення ваги моделі.  І навпаки, чим менший C, тим більший ефект регуляризації.


 Налаштуємо параметр C (сила регуляризації) для кожного типу регуляризації.

In [ ]:
# Ridge
from sklearn.linear_model import Ridge
log_reg = LogisticRegression(solver='liblinear')
log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_valid)

print('F1 score:', f1_score(y_valid, y_pred, average = "weighted"))

In [ ]:
from sklearn.model_selection import GridSearchCV

C_values = {'C': np.logspace(-3, 3, 10)}
logreg_grid = GridSearchCV(log_reg, C_values, cv=5, scoring='f1')
logreg_grid.fit(X_train, y_train)

In [ ]:
print(logreg_grid.best_params_)
print(logreg_grid.best_score_)

In [ ]:
results_df = pd.DataFrame(logreg_grid.cv_results_)
plt.plot(results_df['param_C'], results_df['mean_test_score'])

plt.xlabel('C')
plt.ylabel('Test accuracy')
plt.title('Validation curve')
plt.show()

In [ ]:
# Lasso
log_reg1 = LogisticRegression(max_iter=1000, solver='liblinear', penalty='l1')
log_reg1.fit(X_train, y_train)
y_pred = log_reg1.predict(X_valid)

print('F1 score:', f1_score(y_valid, y_pred, average = "weighted"))

In [ ]:
from sklearn.model_selection import GridSearchCV

C_values = {'C': np.logspace(-3, 3, 10)}
logreg1_grid = GridSearchCV(log_reg1, C_values, cv=5, scoring='f1')
logreg1_grid.fit(X_train, y_train)

In [ ]:
print(logreg1_grid.best_params_)
print(logreg1_grid.best_score_)

In [ ]:
results_df = pd.DataFrame(logreg1_grid.cv_results_)
plt.plot(results_df['param_C'], results_df['mean_test_score'])

plt.xlabel('C')
plt.ylabel('Test accuracy')
plt.title('Validation curve')
plt.show()

In [7]:
#elasticnet
# fix ----- liblinear solver only supports l1/l2, l2 is by default and tried in previous cell, decreased max_iter for faster run
# log_reg2 = LogisticRegression(max_iter=1000, solver='liblinear', penalty='elasticnet')
log_reg2 = LogisticRegression(max_iter=10, solver='liblinear', penalty='l1')
log_reg2.fit(X_train, y_train)
y_pred = log_reg2.predict(X_valid)

print('F1 score:', f1_score(y_valid, y_pred, average = "weighted"))

F1 score: 0.9020291436650468


/usr/local/lib/python3.10/dist-packages/sklearn/svm/_base.py:1244: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [ ]:
C_values = {'C': np.logspace(-3, 3, 10)}
logreg2_grid = GridSearchCV(log_reg2, C_values, cv=5, scoring='f1')
logreg2_grid.fit(X_train, y_train)

In [ ]:
print(logreg2_grid.best_params_)
print(logreg2_grid.best_score_)

In [ ]:
results_df = pd.DataFrame(logreg2_grid.cv_results_)
plt.plot(results_df['param_C'], results_df['mean_test_score'])

plt.xlabel('C')
plt.ylabel('Test accuracy')
plt.title('Validation curve')
plt.show()

In [ ]:
best_scores = [logreg_grid.best_score_, logreg1_grid.best_score_, logreg2_grid.best_score_]
model_names = ['Lasso', 'Ridge', 'ElasticNet']

plt.figure(figsize=(8, 6))
plt.show()

for i in range(len(best_scores)):
    print(f"{model_names[i]} = {best_scores[i]}")